In [0]:
from pyspark.sql import Row

In [0]:
#Assigning catalog and schema values to variables
catalog="practice_db"
schema="bronze"

In [0]:
#Reading Metadata from information schema and storing it in dataframe
df_tables = spark.sql(f"""select table_name from {catalog}.information_schema.tables where table_schema='{schema}'""")

In [0]:
df_tables.display()

In [0]:
#converting dataframe into python list
tables_list= []
for row in df_tables.collect():
    tables_list.append(row.table_name)
tables_list

In [0]:
#selecting count from the tables
results=[]
for row in tables_list:
    full_table_name = f"{catalog}.{schema}.{row}"

    print(f"counting rows")
#row objects : spark cluster memory to python driver memory. list of row objects
    try:
        row_count=spark.sql(f"select count(*) as cnt from {full_table_name}").collect()[0]["cnt"]

        results.append(
            Row
            (table_name=full_table_name, row_count=row_count
            )
        )
    except Exception as e:
        print(f"FAILED: {full_table_name}")
        results.append(
            Row
            (
                table_name=full_table_name, row_count=None
            )
        )
#converting results list of row objects to dataframe
counts_df=spark.createDataFrame(results)
display(counts_df)